# Operational Data, Tools & Function Calling

This notebook creates synthetic airport operational telemetry for SFO, LAX, and JFK.
The dataset provides the operational evidence required by the AI to investigate airport issues.
The data will later be exposed through Python tools so the LLM can retrieve metrics when needed.
The workflow will demonstrate structured tool inputs, tool outputs, and function calling.

In [1]:
# Certificate issue resolve:
import os

os.environ['REQUESTS_CA_BUNDLE'] = '/etc/ssl/certs/ca-certificates.crt'
os.environ['SSL_CERT_FILE'] = '/etc/ssl/certs/ca-certificates.crt'

import sys
import pandas as pd
from google import genai
from google.genai import types
from dotenv import load_dotenv

# Add the project root to Python's import path
sys.path.append("..")

# Load the Gemini API key
load_dotenv()
api_key = os.getenv("GEMINI_API_KEY")

# Create the Gemini client
client = genai.Client(api_key=api_key)

from src.tools import (
    get_airport_metrics,
    calculate_driver_incentive,
    trigger_surge_override
)

print("API key available:", bool(api_key))

API key available: True


## 1. Create Synthetic Airport Telemetry

Synthetic telemetry is created for SFO, LAX, and JFK.
The dataset contains completion rate, ETA, driver supply, cancellations, queue size,
surge multiplier, request volume, and timestamp.
These metrics represent the operational information that the AI will access through tools.

In [2]:
import pandas as pd

# Create synthetic airport telemetry for the three supported airports
data = [
    ["SFO", 0.71, 18, 420, 0.19, 180, 1.2, 620, "2026-09-15 10:00:00"],
    ["SFO", 0.78, 16, 450, 0.16, 150, 1.2, 580, "2026-09-15 11:00:00"],
    ["SFO", 0.86, 12, 500, 0.10, 100, 1.1, 500, "2026-09-15 12:00:00"],

    ["LAX", 0.76, 20, 380, 0.18, 210, 1.3, 700, "2026-09-15 10:00:00"],
    ["LAX", 0.82, 17, 410, 0.14, 170, 1.3, 650, "2026-09-15 11:00:00"],
    ["LAX", 0.87, 13, 460, 0.09, 110, 1.2, 560, "2026-09-15 12:00:00"],

    ["JFK", 0.74, 19, 390, 0.17, 190, 1.2, 680, "2026-09-15 10:00:00"],
    ["JFK", 0.81, 16, 430, 0.13, 150, 1.2, 620, "2026-09-15 11:00:00"],
    ["JFK", 0.88, 12, 480, 0.08, 90, 1.1, 540, "2026-09-15 12:00:00"]
]

# Define the required telemetry columns
columns = [
    "airport_code",
    "completion_rate",
    "average_eta",
    "active_drivers",
    "driver_cancellation_rate",
    "queue_size",
    "surge_multiplier",
    "request_volume",
    "timestamp"
]

# Create the DataFrame from the synthetic records
airport_metrics = pd.DataFrame(data, columns=columns)

# Convert timestamp text into datetime values
airport_metrics["timestamp"] = pd.to_datetime(airport_metrics["timestamp"])

airport_metrics

,airport_code,completion_rate,average_eta,active_drivers,driver_cancellation_rate,queue_size,surge_multiplier,request_volume,timestamp
0,SFO,0.71,18,420,0.19,180,1.2,620,2026-09-15 10:00:00
1,SFO,0.78,16,450,0.16,150,1.2,580,2026-09-15 11:00:00
2,SFO,0.86,12,500,0.10,100,1.1,500,2026-09-15 12:00:00
3,LAX,0.76,20,380,0.18,210,1.3,700,2026-09-15 10:00:00
4,LAX,0.82,17,410,0.14,170,1.3,650,2026-09-15 11:00:00
5,LAX,0.87,13,460,0.09,110,1.2,560,2026-09-15 12:00:00
6,JFK,0.74,19,390,0.17,190,1.2,680,2026-09-15 10:00:00
7,JFK,0.81,16,430,0.13,150,1.2,620,2026-09-15 11:00:00
8,JFK,0.88,12,480,0.08,90,1.1,540,2026-09-15 12:00:00


In [3]:
# Check the dataset structure and data types
airport_metrics.info()

# Check for missing values in the telemetry
print("\nMissing values:")
print(airport_metrics.isnull().sum())

# Display the number of records available for each airport
print(airport_metrics["airport_code"].value_counts())

<class 'pandas.DataFrame'>
RangeIndex: 9 entries, 0 to 8
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   airport_code              9 non-null      str           
 1   completion_rate           9 non-null      float64       
 2   average_eta               9 non-null      int64         
 3   active_drivers            9 non-null      int64         
 4   driver_cancellation_rate  9 non-null      float64       
 5   queue_size                9 non-null      int64         
 6   surge_multiplier          9 non-null      float64       
 7   request_volume            9 non-null      int64         
 8   timestamp                 9 non-null      datetime64[us]
dtypes: datetime64[us](1), float64(3), int64(4), str(1)
memory usage: 780.0 bytes

Missing values:
airport_code                0
completion_rate             0
average_eta                 0
active_drivers              0
driver_can

In [4]:
from pathlib import Path

# Define the required output location
output_path = Path("../data/airport_metrics.csv")

# Save the synthetic telemetry as a CSV file
airport_metrics.to_csv(output_path, index=False)

print(f"Saved dataset to: {output_path}")

# Reload the saved CSV to confirm that it was written correctly
test_data = pd.read_csv(output_path)

print(test_data.head())
print("\nRows:", len(test_data))

Saved dataset to: ../data/airport_metrics.csv
  airport_code  completion_rate  average_eta  active_drivers  \
0          SFO             0.71           18             420   
1          SFO             0.78           16             450   
2          SFO             0.86           12             500   
3          LAX             0.76           20             380   
4          LAX             0.82           17             410   

   driver_cancellation_rate  queue_size  surge_multiplier  request_volume  \
0                      0.19         180               1.2             620   
1                      0.16         150               1.2             580   
2                      0.10         100               1.1             500   
3                      0.18         210               1.3             700   
4                      0.14         170               1.3             650   

             timestamp  
0  2026-09-15 10:00:00  
1  2026-09-15 11:00:00  
2  2026-09-15 12:00:00  
3  202

## 2. Test Operational Tools

The operational tools provide controlled access to airport telemetry and operational actions.
Each tool accepts structured inputs and returns structured outputs.
The tools are tested independently before connecting them to the LLM.

In [5]:
import sys

# Add the project root to Python's import path
sys.path.append("..")

from src.tools import (
    get_airport_metrics,
    calculate_driver_incentive,
    trigger_surge_override
)

In [6]:
# Retrieve the latest operational metrics for SFO
sfo_metrics = get_airport_metrics("SFO")

print(sfo_metrics)

{'status': 'success', 'airport_code': 'SFO', 'completion_rate': 0.86, 'average_eta': 12, 'active_drivers': 500, 'driver_cancellation_rate': 0.1, 'queue_size': 100, 'surge_multiplier': 1.1}


In [7]:
# Test how the tool handles an unsupported airport
print(get_airport_metrics("ABC"))

{'status': 'error', 'message': 'Invalid airport code: ABC'}


In [8]:
# Calculate the incentive for 50 drivers under high shortage severity
incentive = calculate_driver_incentive(50, "high")

print(incentive)

{'status': 'success', 'driver_count': 50, 'severity_level': 'high', 'recommended_incentive': 20, 'estimated_total_cost': 1000}


In [9]:
# Test validation for a negative driver count
print(calculate_driver_incentive(-10, "high"))

{'status': 'error', 'message': 'driver_count cannot be negative'}


In [10]:
# Test the mock surge override tool
override = trigger_surge_override(
    "SFO",
    1.5,
    "Low completion rate and insufficient driver supply"
)

print(override)

{'status': 'success', 'airport_code': 'SFO', 'new_multiplier': 1.5, 'reason': 'Low completion rate and insufficient driver supply', 'message': 'Surge override executed successfully (mock)'}


In [11]:
# Test validation for an invalid surge multiplier
print(
    trigger_surge_override(
        "SFO",
        5.0,
        "Test invalid multiplier"
    )
)

{'status': 'error', 'message': 'Surge multiplier must be between 1.0x and 2.0x'}


## 3. Test Tool Execution

Each operational tool was tested using both valid and invalid inputs.

- `get_airport_metrics()` successfully retrieved the latest operational metrics for valid airport codes and returned an error for an invalid airport code.
- `calculate_driver_incentive()` successfully calculated the recommended incentive and estimated cost for valid inputs and rejected invalid driver counts or severity levels.
- `trigger_surge_override()` successfully performed the mock surge override for valid inputs and rejected invalid surge multipliers.

These tests confirm that the tools return structured results and provide basic error handling before being connected to the LLM function-calling workflow.

## 4. Define Function-Calling Tool Schemas

The LLM needs structured descriptions of the available operational tools before it can decide when to use them.

Each tool schema defines its name, purpose, required parameters, and parameter types. This allows the LLM to select the appropriate tool and provide the required inputs instead of inventing operational metrics.

In [12]:
from google.genai import types

# Define the tools that Gemini can choose from
tools = [
    {
        "function_declarations": [
            {
                "name": "get_airport_metrics",
                "description": "Get the latest operational metrics for an airport.",
                "parameters": {
                    "type": "OBJECT",
                    "properties": {
                        "airport_code": {
                            "type": "STRING",
                            "description": "Airport code: SFO, LAX, or JFK."
                        }
                    },
                    "required": ["airport_code"]
                }
            },
            {
                "name": "calculate_driver_incentive",
                "description": "Calculate the recommended driver incentive and estimated total cost.",
                "parameters": {
                    "type": "OBJECT",
                    "properties": {
                        "driver_count": {
                            "type": "NUMBER",
                            "description": "Number of drivers to incentivize."
                        },
                        "severity_level": {
                            "type": "STRING",
                            "description": "Driver shortage severity: low, medium, or high."
                        }
                    },
                    "required": ["driver_count", "severity_level"]
                }
            },
            {
                "name": "trigger_surge_override",
                "description": "Execute a mock surge multiplier override for an airport.",
                "parameters": {
                    "type": "OBJECT",
                    "properties": {
                        "airport_code": {
                            "type": "STRING",
                            "description": "Airport code: SFO, LAX, or JFK."
                        },
                        "new_multiplier": {
                            "type": "NUMBER",
                            "description": "New surge multiplier between 1.0 and 2.0."
                        },
                        "reason": {
                            "type": "STRING",
                            "description": "Operational reason for the override."
                        }
                    },
                    "required": ["airport_code", "new_multiplier", "reason"]
                }
            }
        ]
    }
]

print("Tool schemas defined:", len(tools[0]["function_declarations"]))

Tool schemas defined: 3


## 5. Connect Gemini to the Operational Tools

The tool schemas are now provided to Gemini so the model knows which operational actions are available.

The system instruction tells Gemini to use a tool whenever the user requests current airport operational information. It also prevents the model from inventing operational metrics when the required data should come from a tool.

In [13]:
# Map tool names from Gemini to the actual Python functions
tool_map = {
    "get_airport_metrics": get_airport_metrics,
    "calculate_driver_incentive": calculate_driver_incentive,
    "trigger_surge_override": trigger_surge_override
}

# Tell Gemini when it should use the available tools
system_instruction = """
You are an Airport Operations AI Copilot.

Use the available tools whenever the user asks for current airport
operational metrics, driver incentives, or a surge override.

Never invent operational metrics.
Use tool results as the source of operational data.
If a tool returns an error, explain the error to the user.
"""

# Test whether Gemini identifies the required tool
user_query = "What's happening at SFO?"

response = client.models.generate_content(
    model="gemini-3.5-flash-lite",
    contents=user_query,
    config=types.GenerateContentConfig(
        system_instruction=system_instruction,
        tools=tools
    )
)

print(response)

sdk_http_response=HttpResponse(
  headers=<dict len=12>
) candidates=[Candidate(
  content=Content(
    parts=[
      Part(
        function_call=FunctionCall(
          args={
            'airport_code': 'SFO'
          },
          id='call_96150',
          name='get_airport_metrics'
        ),
        thought_signature=b'\x12^\n\\\x01\x11M2\x0f3\xb7\xcf\x97\xba\xc0\xd7K\xe6\x81`m\xb2\xf6\xd6-\xd3\xa4\xbe\x87\x91\xdbO\x7f\xbe8\x0f\x11\x96I\xfbS\xe4\xd3a\xd7S>\x90\x87HF"\x8b\xeeF\x88\xfeQ\xcbN\x12\xd8y\xc6\xc9\n\xca\xdd\xbb\xe7\x00BLG\xec"l\xfc\xa0\xca\xe7\x1f\x1e\xb4G\xb4\xec\x98gc3\xebjo\xee~'
      ),
    ],
    role='model'
  ),
  finish_reason=<FinishReason.STOP: 'STOP'>,
  index=0
)] create_time=None model_version='gemini-3.5-flash-lite' prompt_feedback=None response_id='EhiqavWHKIqng8UPypykqA4' usage_metadata=GenerateContentResponseUsageMetadata(
  candidates_token_count=21,
  prompt_token_count=371,
  prompt_tokens_details=[
    ModalityTokenCount(
      modality=<MediaModali

## 6. Execute the Selected Tool

Gemini has identified the appropriate operational tool and provided the required input.

The function call is now passed to the corresponding Python function. The returned structured result will then be provided back to Gemini so the model can generate a final response based on the actual operational data.

In [14]:
# Get the function call selected by Gemini
function_call = response.function_calls[0]

# Extract the tool name and arguments
function_name = function_call.name
arguments = function_call.args

print("Selected tool:", function_name)
print("Arguments:", arguments)

# Execute the selected Python tool
tool_result = tool_map[function_name](**arguments)

print("\nTool result:")
print(tool_result)

Selected tool: get_airport_metrics
Arguments: {'airport_code': 'SFO'}

Tool result:
{'status': 'success', 'airport_code': 'SFO', 'completion_rate': 0.86, 'average_eta': 12, 'active_drivers': 500, 'driver_cancellation_rate': 0.1, 'queue_size': 100, 'surge_multiplier': 1.1}


## 7. Generate the Final Tool-Based Response

The selected tool has now returned the actual airport metrics.

The tool result is sent back to Gemini along with the original function call. Gemini can use this verified operational data to generate the final response instead of relying on assumptions or general knowledge.

In [16]:
# Build the conversation with Gemini's function call and the tool result
contents = [
    types.Content(
        role="user",
        parts=[types.Part.from_text(text=user_query)]
    ),
    response.candidates[0].content,
    types.Content(
        role="user",
        parts=[
            types.Part.from_function_response(
                name=function_name,
                response={"result": tool_result}
            )
        ]
    )
]

# Ask Gemini to generate the final response using the tool result
final_response = client.models.generate_content(
    model="gemini-3.5-flash-lite",
    contents=contents,
    config=types.GenerateContentConfig(
        system_instruction=system_instruction,
        tools=tools
    )
)

print(final_response.text)

Here are the current operational metrics for SFO:

* **Status:** Success
* **Active Drivers:** 500
* **Queue Size:** 100
* **Average ETA:** 12 minutes
* **Completion Rate:** 86%
* **Driver Cancellation Rate:** 10%
* **Surge Multiplier:** 1.1x


## 8. Test Function-Calling Decisions

Different operational questions should cause Gemini to select different tools.

These tests verify that the model can distinguish between retrieving airport metrics, calculating driver incentives, and executing a surge override.

In [17]:
# Ask Gemini a question that requires the incentive calculation tool
user_query_2 = "How much would it cost to incentivize 50 drivers at high severity?"

response_2 = client.models.generate_content(
    model="gemini-3.5-flash-lite",
    contents=user_query_2,
    config=types.GenerateContentConfig(
        system_instruction=system_instruction,
        tools=tools
    )
)

# Display the selected function call
print(response_2.function_calls)

[FunctionCall(
  args={
    'driver_count': 50,
    'severity_level': 'high'
  },
  id='call_125117',
  name='calculate_driver_incentive'
)]


In [18]:
# Ask Gemini a question that requires the surge override tool
user_query_3 = "Set SFO surge to 1.5x because completion rate is low."

response_3 = client.models.generate_content(
    model="gemini-3.5-flash-lite",
    contents=user_query_3,
    config=types.GenerateContentConfig(
        system_instruction=system_instruction,
        tools=tools
    )
)

# Display the selected function call
print(response_3.function_calls)

[FunctionCall(
  args={
    'airport_code': 'SFO',
    'new_multiplier': 1.5,
    'reason': 'Completion rate is low'
  },
  id='call_193067',
  name='trigger_surge_override'
)]


## 10. Test Tool Execution and Error Handling

The function-calling workflow must handle both successful and failed tool executions.

The following tests execute the functions selected by Gemini and verify that invalid inputs produce structured error responses rather than breaking the workflow.

In [19]:
# Execute the incentive tool selected by Gemini
function_call_2 = response_2.function_calls[0]

tool_result_2 = tool_map[function_call_2.name](**function_call_2.args)

print("Tool:", function_call_2.name)
print("Result:", tool_result_2)

Tool: calculate_driver_incentive
Result: {'status': 'success', 'driver_count': 50, 'severity_level': 'high', 'recommended_incentive': 20, 'estimated_total_cost': 1000}


In [20]:
# Execute the surge override tool selected by Gemini
function_call_3 = response_3.function_calls[0]

tool_result_3 = tool_map[function_call_3.name](**function_call_3.args)

print("Tool:", function_call_3.name)
print("Result:", tool_result_3)

Tool: trigger_surge_override
Result: {'status': 'success', 'airport_code': 'SFO', 'new_multiplier': 1.5, 'reason': 'Completion rate is low', 'message': 'Surge override executed successfully (mock)'}


In [21]:
# Test the tool's handling of an invalid surge multiplier
error_result = trigger_surge_override(
    "SFO",
    5.0,
    "Test invalid multiplier"
)

print(error_result)

{'status': 'error', 'message': 'Surge multiplier must be between 1.0x and 2.0x'}


## 11. Final Function-Calling Validation

The completed workflow is validated using representative operational questions.

The tests confirm that Gemini can identify when operational data is required, select the appropriate tool, execute it successfully, and return structured results. Invalid inputs are also handled without breaking the workflow.

In [22]:
# Validate that each operational question selects the expected tool
validation_queries = [
    ("What's happening at SFO?", "get_airport_metrics"),
    ("How much would it cost to incentivize 50 drivers at high severity?",
     "calculate_driver_incentive"),
    ("Set SFO surge to 1.5x because completion rate is low.",
     "trigger_surge_override")
]

for query, expected_tool in validation_queries:
    # Ask Gemini to select a tool for the query
    result = client.models.generate_content(
        model="gemini-3.5-flash-lite",
        contents=query,
        config=types.GenerateContentConfig(
            system_instruction=system_instruction,
            tools=tools
        )
    )

    selected_tool = result.function_calls[0].name

    print(f"Query: {query}")
    print(f"Selected tool: {selected_tool}")
    print(f"Expected tool: {expected_tool}")
    print(f"Match: {selected_tool == expected_tool}\n")

Query: What's happening at SFO?
Selected tool: get_airport_metrics
Expected tool: get_airport_metrics
Match: True

Query: How much would it cost to incentivize 50 drivers at high severity?
Selected tool: calculate_driver_incentive
Expected tool: calculate_driver_incentive
Match: True

Query: Set SFO surge to 1.5x because completion rate is low.
Selected tool: trigger_surge_override
Expected tool: trigger_surge_override
Match: True



## 12. Summary

This notebook implemented the operational tool layer for the Airport Operations AI Copilot.

- Created a synthetic airport telemetry dataset for SFO, LAX, and JFK.
- Built three operational tools for metrics, driver incentives, and surge overrides.
- Defined structured function schemas for LLM tool selection.
- Connected Gemini to the operational tools using function calling.
- Executed selected tools and returned their results to the LLM.
- Added validation and error handling for invalid tool inputs.
- Verified that Gemini selects the appropriate tool for representative operational questions.

The completed workflow demonstrates how an LLM can access structured operational data through controlled tools rather than inventing operational metrics.